# Session 3: Data Quality Assessment
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Identify and measure missing values
- Detect duplicate records
- Spot outliers using the IQR method and z-scores
- Detect inconsistent categories and data entry issues
- Produce a data quality report

---

## 1. Load the Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("World_Dev_Indicators.csv")
print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 2. Missing Values

Missing values appear as `NaN` (Not a Number) in pandas.
Before cleaning data, you need to know:
- **Which columns** have missing values?
- **How many** are missing?
- **What percentage** are missing?

In [ ]:
# Is each cell null? Returns a DataFrame of True/False
df.isnull().head()

In [ ]:
# Total missing values per column
null_counts = df.isnull().sum()
print("Missing values per column:")
print(null_counts)

In [ ]:
# Percentage missing — more interpretable than raw counts
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
print("\nPercentage missing per column:")
print(null_pct)

In [ ]:
# Build a clear missing values summary table
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %":    (df.isnull().sum() / len(df) * 100).round(2),
    "Data Type":    df.dtypes
})
missing_summary = missing_summary[missing_summary["Missing Count"] > 0]
missing_summary = missing_summary.sort_values("Missing %", ascending=False)
print("Columns with missing values:")
missing_summary

### 2.1 Visualising Missingness Patterns

Sometimes missingness is not random — it clusters by year, region, or income group.
Let us check whether nulls in GDP are concentrated in certain years.

In [ ]:
# Are GDP nulls concentrated in specific years?
gdp_col = "GDP (current US$)"
null_by_year = df[df[gdp_col].isnull()].groupby("Year").size()
print("Missing GDP values by year:")
print(null_by_year)

In [ ]:
# Are GDP nulls concentrated in specific regions?
null_by_region = df[df[gdp_col].isnull()].groupby("Region").size().sort_values(ascending=False)
print("Missing GDP values by region:")
print(null_by_region)

In [ ]:
# Are GDP nulls concentrated in certain income groups?
null_by_income = df[df[gdp_col].isnull()]["Income Group"].value_counts()
print("Missing GDP values by income group:")
print(null_by_income)

## 3. Duplicate Records

Duplicates inflate counts and skew statistics. They often result from
data entry errors or merging datasets that overlap.

In [ ]:
# Check for fully duplicate rows (every column matches)
num_duplicates = df.duplicated().sum()
print(f"Fully duplicate rows: {num_duplicates}")

# Check for duplicates on key columns (Country + Year should be unique)
key_dups = df.duplicated(subset=["Country", "Year"]).sum()
print(f"Duplicate (Country, Year) combinations: {key_dups}")

In [ ]:
# If duplicates exist, see them
if key_dups > 0:
    dup_mask = df.duplicated(subset=["Country","Year"], keep=False)
    print("\nRows with duplicate Country + Year:")
    print(df[dup_mask].sort_values(["Country","Year"]).head(10))
else:
    print("No (Country, Year) duplicates found — data looks clean on this key!")

## 4. Outlier Detection

Outliers are values that are unusually far from the rest of the data.
They might be:
- **Legitimate extremes** (very large countries, very rich nations)
- **Data errors** (e.g., a value entered as 1,000x too large)

### 4.1 The IQR Method

The Interquartile Range (IQR) is the spread between the 25th and 75th percentile.
Values outside `[Q1 - 1.5*IQR, Q3 + 1.5*IQR]` are flagged as potential outliers.

In [ ]:
def flag_outliers_iqr(series):
    """Return a boolean mask: True where values are outliers by IQR rule."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

# Apply to GDP per capita
gdppc_col = "GDP per capita (current US$)"
outlier_mask = flag_outliers_iqr(df[gdppc_col].dropna())
print(f"GDP per capita outliers: {outlier_mask.sum()} rows")

In [ ]:
# Look at the high-end outliers
high_outliers = df.loc[
    flag_outliers_iqr(df[gdppc_col]) & df[gdppc_col].notna()
].sort_values(gdppc_col, ascending=False)

print("Top 10 high-end GDP per capita outliers:")
print(high_outliers[["Year","Country","Region","Income Group", gdppc_col]].head(10))

### 4.2 The Z-Score Method

Z-scores measure how many standard deviations a value is from the mean.
Values with |z| > 3 are typically flagged as outliers.

In [ ]:
# Z-score outlier detection
life_col = "Life expectancy at birth, total (years)"
df_clean = df[df[life_col].notna()].copy()

mean_le = df_clean[life_col].mean()
std_le  = df_clean[life_col].std()
df_clean["life_zscore"] = (df_clean[life_col] - mean_le) / std_le

# Flag outliers
le_outliers = df_clean[df_clean["life_zscore"].abs() > 3]
print(f"Life expectancy outliers (|z| > 3): {len(le_outliers)} rows")
print()
le_outliers[["Year","Country","Region",life_col,"life_zscore"]].sort_values("life_zscore")

## 5. Detecting Inconsistent Categories

String columns often have subtle inconsistencies — leading spaces, mixed case,
or slightly different spellings of the same value.

In [ ]:
# Check for leading/trailing whitespace in Region
regions_raw = df["Region"].unique()
print("Raw Region values (note any extra spaces):")
for r in sorted(regions_raw):
    print(repr(r))   # repr() shows hidden whitespace

In [ ]:
# Check Income Group for inconsistencies
print("Income Group values:")
for v in sorted(df["Income Group"].unique()):
    print(repr(v))

In [ ]:
# Check for mixed case issues in Country names
# Are there countries where the same country appears with different casings?
df["Country_lower"] = df["Country"].str.lower().str.strip()
lower_counts = df.groupby("Country_lower")["Country"].nunique()
casing_issues = lower_counts[lower_counts > 1]
if len(casing_issues) > 0:
    print("Countries with potential casing inconsistencies:")
    print(casing_issues)
else:
    print("No casing inconsistencies found in Country column.")

## 6. Building a Data Quality Report

Let us combine everything into a single function that summarises data quality.

In [ ]:
def data_quality_report(df):
    """Print a comprehensive data quality report."""
    print("=" * 60)
    print("  DATA QUALITY REPORT")
    print("=" * 60)
    print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")
    print()

    # Missing values
    print("--- MISSING VALUES ---")
    nulls = df.isnull().sum()
    null_pct = (nulls / len(df) * 100).round(1)
    for col in df.columns:
        if nulls[col] > 0:
            print(f"  {col:<48} {nulls[col]:>5} missing ({null_pct[col]}%)")
    if nulls.sum() == 0:
        print("  No missing values found!")

    # Duplicates
    print()
    print("--- DUPLICATES ---")
    total_dups = df.duplicated().sum()
    key_dups   = df.duplicated(subset=["Country","Year"]).sum() if "Country" in df.columns and "Year" in df.columns else "N/A"
    print(f"  Fully duplicate rows:        {total_dups}")
    print(f"  Duplicate (Country,Year) rows: {key_dups}")

    # Outliers (IQR) for numeric columns
    print()
    print("--- OUTLIERS (IQR method) ---")
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        s = df[col].dropna()
        if len(s) == 0:
            continue
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        out_count = ((s < Q1 - 1.5*IQR) | (s > Q3 + 1.5*IQR)).sum()
        if out_count > 0:
            print(f"  {col:<48} {out_count:>5} potential outliers")
    print("=" * 60)

data_quality_report(df)

## 7. Summary Cheat Sheet

| Check | Code |
|-------|------|
| Count NaNs per column | `df.isnull().sum()` |
| Percentage missing | `df.isnull().sum() / len(df) * 100` |
| Find fully duplicate rows | `df.duplicated().sum()` |
| Find key-column duplicates | `df.duplicated(subset=['A','B']).sum()` |
| IQR outlier detection | `(s < Q1-1.5*IQR) \| (s > Q3+1.5*IQR)` |
| Z-score outlier detection | `(s - mean) / std > 3` |
| Show hidden whitespace | `repr(value)` |

---

## 8. Exercises

1. Which year has the most missing `Life expectancy` values?
2. Are there any countries that appear in some years but not all 20 years? Count observations per country with `value_counts()`.
3. Find outliers in `Suicide mortality rate (per 100,000 population)` using the IQR method. Which countries appear most often?
4. **Bonus:** Check whether `Region` values have leading or trailing spaces. How many rows are affected?

In [ ]:
# Your answers here:
# Exercise 1

# Exercise 2

# Exercise 3

# Exercise 4 (bonus)
